In [3]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import time
from datetime import datetime

import matplotlib.pyplot as plt
from ase import units
from ase.io import read
from ase.build import molecule
from ase.optimize import BFGS
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.nose_hoover_chain import NoseHooverChainNVT
from ase.geometry import get_distances
from ase.units import fs
from fairchem.core import pretrained_mlip, FAIRChemCalculator

In [4]:
# ── Calculator ────────────────────────────────────────────────────────────────
# initialize
predictor = pretrained_mlip.get_predict_unit('uma-s-1p1', device='cpu')
calc = FAIRChemCalculator(predictor, task_name='omat')

In [5]:
# ── Load MOF ──────────────────────────────────────────────────────────────────
mof_74 = read('mg_mof74.cif')

# ── Reference energies — always recomputed with the current calc ──────────────
# FIX: The original if/else reused E_mof/E_co2 from a previous session when
# structures were already in memory. Absolute energies are NOT transferable
# across calculator instances — this caused the +5 eV E_bind drift.
print('Computing E_mof...')
mof_ref = mof_74.copy()
mof_ref.calc = calc
BFGS(mof_ref, logfile='opt_mof_74.log').run(fmax=0.01)
E_mof = mof_ref.get_potential_energy()

print('Computing E_co2...')
co2_ref = molecule('CO2')
co2_ref.set_pbc(False)
co2_ref.center(vacuum=10.0)
co2_ref.calc = calc
BFGS(co2_ref, logfile='opt_co2.log').run(fmax=0.01)
E_co2 = co2_ref.get_potential_energy()

print(f'E(MOF) = {E_mof:.6f} eV')   # expect ~ -1177 eV
print(f'E(CO2) = {E_co2:.6f} eV')   # expect ~ -22.6 eV

Computing E_mof...
Computing E_co2...
E(MOF) = -1177.415312 eV
E(CO2) = -22.596883 eV


In [6]:
# ── Load pre-placed structure from CIF ───────────────────────────────────────
from ase.io import read

system_md = read('mof_co2_initial.cif')
system_md.set_pbc(True)
system_md.calc = calc

# Identify framework vs CO2 atoms
# CO2 is the last 3 atoms (C + 2 O) added in the GUI
n_framework = len(system_md) - 3
print(f'System: {len(system_md)} atoms ({n_framework} MOF + 3 CO2)')
print(f'Last 3 atoms (CO2): {[system_md[i].symbol for i in range(n_framework, len(system_md))]}')

# ── Pre-relax ─────────────────────────────────────────────────────────────────
print('Pre-relaxing...')
t0 = time.time()
BFGS(system_md, logfile='md_prerelax.log').run(fmax=0.001)
print(f'Done in {time.time()-t0:.1f} s')

E_bind_prerelax = system_md.get_potential_energy() - E_mof - E_co2
print(f'E_bind (pre-MD) = {E_bind_prerelax:.4f} eV')
if E_bind_prerelax > 0:
    raise ValueError('E_bind positive before MD — check CO2 placement in CIF.')

System: 165 atoms (162 MOF + 3 CO2)
Last 3 atoms (CO2): ['C', 'O', 'O']
Pre-relaxing...
Done in 1163.3 s
E_bind (pre-MD) = -0.5862 eV


In [7]:
# ── MD runner ─────────────────────────────────────────────────────────────────
best_mg_idx   = 'gui'   # used in trajectory filename only
static_E_bind = system_md.get_potential_energy() - E_mof - E_co2

def run_md(system_in, temperature_K, n_steps=1000, timestep_fs=0.5,
           tdamp_fs=100, traj_every=10, log_every=50):

    atoms = system_in.copy()
    atoms.calc = calc

    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
    Stationary(atoms)

    ts        = datetime.now().strftime('%Y%m%d_%H%M%S')
    traj_file = f'md_site{best_mg_idx}_{temperature_K}K_{ts}.traj'
    log_file  = f'md_site{best_mg_idx}_{temperature_K}K_{ts}.log'

    dyn = NoseHooverChainNVT(
        atoms,
        timestep      = timestep_fs * fs,
        temperature_K = temperature_K,
        tdamp         = tdamp_fs * fs,
        trajectory    = traj_file,
        logfile       = log_file,
    )

    records = []

    def log_step():
        step  = dyn.get_number_of_steps()
        E_pot = atoms.get_potential_energy()
        E_b   = E_pot - E_mof - E_co2
        records.append((step, E_pot, E_b))
        if step % log_every == 0:
            print(f'  step {step:4d} | T = {atoms.get_temperature():6.1f} K | '
                  f'E_pot = {E_pot:.4f} eV | E_bind = {E_b:.4f} eV')

    dyn.attach(log_step, interval=traj_every)

    print(f'\n{"="*55}')
    print(f'  MD  |  T = {temperature_K} K  |  {n_steps} steps  |  {len(atoms)} atoms')
    print(f'{"="*55}')

    t_start = time.time()
    dyn.run(steps=n_steps)
    elapsed = time.time() - t_start

    print(f'\n  Done in {elapsed:.1f} s  ({elapsed/n_steps*1000:.0f} ms/step)')
    print(f'  Trajectory saved to {traj_file}')

    return atoms, records, traj_file

In [8]:
# ── Temperature sweep ─────────────────────────────────────────────────────────
temperatures = range(300, 1050, 50)

all_records   = {}
all_trajs     = {}
final_structs = {}

for T in temperatures:
    print(f'\nStarting MD at {T} K...')
    final_T, records_T, traj_T = run_md(
        system_md,            # always from pre-relaxed structure
        temperature_K=T
    )
    all_records[T]   = records_T
    all_trajs[T]     = traj_T
    final_structs[T] = final_T

print('\nAll temperatures complete!')


Starting MD at 300 K...

  MD  |  T = 300 K  |  1000 steps  |  165 atoms
  step    0 | T =  289.7 K | E_pot = -1200.5984 eV | E_bind = -0.5862 eV
  step   50 | T =  153.7 K | E_pot = -1197.6109 eV | E_bind = 2.4013 eV
  step  100 | T =  124.7 K | E_pot = -1196.7489 eV | E_bind = 3.2632 eV
  step  150 | T =  168.6 K | E_pot = -1197.4283 eV | E_bind = 2.5839 eV
  step  200 | T =  144.9 K | E_pot = -1196.6762 eV | E_bind = 3.3360 eV
  step  250 | T =  169.4 K | E_pot = -1196.8968 eV | E_bind = 3.1154 eV
  step  300 | T =  219.4 K | E_pot = -1197.6176 eV | E_bind = 2.3946 eV
  step  350 | T =  180.9 K | E_pot = -1196.4245 eV | E_bind = 3.5877 eV
  step  400 | T =  183.4 K | E_pot = -1196.1011 eV | E_bind = 3.9111 eV
  step  450 | T =  248.2 K | E_pot = -1197.0800 eV | E_bind = 2.9322 eV
  step  500 | T =  208.1 K | E_pot = -1195.8029 eV | E_bind = 4.2093 eV
  step  550 | T =  201.2 K | E_pot = -1195.2201 eV | E_bind = 4.7921 eV
  step  600 | T =  261.2 K | E_pot = -1196.0758 eV | E_bind =

In [9]:



# ── Summary statistics ────────────────────────────────────────────────────────
for T, records in sorted(all_records.items()):
    E_binds = np.array([r[2] for r in records])
    eq_cut  = len(E_binds) // 5
    prod    = E_binds[eq_cut:]
    print(f'\nT = {T} K  (production = {len(prod)} frames)')
    print(f'  <E_bind> = {prod.mean():.4f} eV')
    print(f'  std      = {prod.std():.4f} eV')
    print(f'  min/max  = {prod.min():.4f} / {prod.max():.4f} eV')


T = 300 K  (production = 81 frames)
  <E_bind> = 4.5472 eV
  std      = 1.1074 eV
  min/max  = 2.3946 / 7.1500 eV

T = 350 K  (production = 81 frames)
  <E_bind> = 5.6761 eV
  std      = 1.1707 eV
  min/max  = 3.6442 / 8.5565 eV

T = 400 K  (production = 81 frames)
  <E_bind> = 6.7229 eV
  std      = 1.3109 eV
  min/max  = 4.0729 / 9.7063 eV

T = 450 K  (production = 81 frames)
  <E_bind> = 7.5787 eV
  std      = 1.4276 eV
  min/max  = 4.6493 / 10.4103 eV

T = 500 K  (production = 81 frames)
  <E_bind> = 8.2431 eV
  std      = 1.6783 eV
  min/max  = 5.3270 / 12.1279 eV

T = 550 K  (production = 81 frames)
  <E_bind> = 9.1196 eV
  std      = 1.9030 eV
  min/max  = 5.3649 / 13.0349 eV

T = 600 K  (production = 81 frames)
  <E_bind> = 9.4511 eV
  std      = 2.1780 eV
  min/max  = 4.8021 / 13.6873 eV

T = 650 K  (production = 81 frames)
  <E_bind> = 10.9490 eV
  std      = 2.2444 eV
  min/max  = 6.7496 / 14.8352 eV

T = 700 K  (production = 81 frames)
  <E_bind> = 11.3604 eV
  std      = 